In [1]:
# ============================================================
# CELL 1: Load IndicBART + reproduce test split + sanity check
# ============================================================
import os, json
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

MODEL_DIR = "/kaggle/input/datasets/siddhanttimalsina/indicbert-model"
DATA_DIR  = "/kaggle/input/datasets/siddhanttimalsina/nepali-text-summarizer/"

# ---------- Inspect config ----------
with open(os.path.join(MODEL_DIR, 'config.json')) as f:
    cfg = json.load(f)
print(f"Architectures      : {cfg.get('architectures')}")
print(f"Model type         : {cfg.get('model_type')}")
print(f"d_model            : {cfg.get('d_model')}")
print(f"Encoder layers     : {cfg.get('encoder_layers', cfg.get('num_hidden_layers'))}")
print(f"Decoder layers     : {cfg.get('decoder_layers', cfg.get('num_hidden_layers'))}")
print(f"Vocab size         : {cfg.get('vocab_size')}")

# ---------- Load ----------
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print(f"\nIndicBART loaded   : {model.num_parameters()/1e6:.1f}M params")
print(f"Tokenizer          : {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")
print(f"decoder_start_tok  : {model.config.decoder_start_token_id}")
print(f"pad/eos            : {model.config.pad_token_id} / {model.config.eos_token_id}")

# ---------- Reproduce EXACT test split from indicbart_train.py ----------
small  = pd.read_csv(DATA_DIR + "small.csv")
medium = pd.read_csv(DATA_DIR + "medium.csv")
large  = pd.read_csv(DATA_DIR + "large.csv")
all_data = pd.concat([small, medium, large], ignore_index=True)
print(f"\nall_data rows (raw): {len(all_data)}")

_, temp_df = train_test_split(all_data, test_size=0.2, random_state=42)
_, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"IndicBART test split: {len(test_df)} rows")

before = len(test_df)
test_df = test_df.dropna(subset=['content', 'summary']).copy()
test_df['content'] = test_df['content'].astype(str).str.strip()
test_df['summary'] = test_df['summary'].astype(str).str.strip()
test_df = test_df[(test_df['content'].str.len() > 0) &
                  (test_df['summary'].str.len() > 0)].reset_index(drop=True)
print(f"After NaN/empty filter: {len(test_df)} rows (dropped {before - len(test_df)})")

test_df[['content', 'summary']].to_csv(
    '/kaggle/working/indicbart_test_raw.csv', index=False, encoding='utf-8')
print("Saved indicbart_test_raw.csv")

# ---------- Sanity generation ----------
sample = test_df.iloc[0]
inputs = tokenizer(
    sample['content'],
    return_tensors='pt', max_length=512, truncation=True
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=128, num_beams=4, early_stopping=True,
    )
pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== Sanity check ===")
print(f"REF  ({len(sample['summary'])} chars): {sample['summary'][:300]}")
print(f"PRED ({len(pred)} chars): {pred[:300]}")

Device: cuda
Architectures      : ['MBartForConditionalGeneration']
Model type         : mbart
d_model            : 1024
Encoder layers     : 6
Decoder layers     : 6
Vocab size         : 64014


Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



IndicBART loaded   : 440.7M params
Tokenizer          : AlbertTokenizer, vocab=64000
decoder_start_tok  : None
pad/eos            : 0 / 64001

all_data rows (raw): 50529
IndicBART test split: 5053 rows
After NaN/empty filter: 5053 rows (dropped 0)
Saved indicbart_test_raw.csv

=== Sanity check ===
REF  (349 chars): स्वास्थ्य बीमा बोर्डले आगामी पाँच वर्षका लागि रणनीतिक योजना सार्वजनिक गरेको छ, जसमा संस्थागत सुदृढीकरण, ऐन नियमको सुधार, र नागरिकको सहभागिता बढाउने कुरालाई प्राथमिकता दिइएको छ। हाल २८ प्रतिशत नागरिक बीमामा आवद्ध भए पनि बीमितले गुणस्तरीय सेवा नपाएको गुनासो बढेको छ। सेवा प्रवाहमा सुधार र नवीकरण दर बढा
PRED (231 chars): <s> २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०८०-८१, २०


In [2]:
# ============================================================
# CELL 1: IndicBART training — 3 epochs, all fixes
# ============================================================
!pip install -q transformers datasets pandas scikit-learn sentencepiece accelerate

import os, time, shutil, transformers
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, Seq2SeqTrainer,
)

print(f"transformers {transformers.__version__}")
os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type != 'cuda':
    raise RuntimeError("Enable GPU: Settings → Accelerator → GPU T4 x2")

# ---------- Base model ----------
BASE = "ai4bharat/IndicBART"
print(f"\nLoading base model: {BASE}")
tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE)
model.gradient_checkpointing_enable()

HI_ID = tokenizer.convert_tokens_to_ids('<2hi>')
if HI_ID == tokenizer.unk_token_id:
    raise RuntimeError("<2hi> not in vocab — wrong tokenizer")
print(f"<2hi> token id: {HI_ID}  ✓")

# ---------- Critical fix: decoder_start_token_id ----------
model.config.decoder_start_token_id = HI_ID
model.config.pad_token_id           = tokenizer.pad_token_id
model.config.eos_token_id           = tokenizer.eos_token_id
print(f"decoder_start_token_id: {model.config.decoder_start_token_id}  ✓")

# ---------- Data ----------
DATA_DIR = "/kaggle/input/datasets/siddhanttimalsina/mt5-dataset/"
if not os.path.exists(DATA_DIR + "small.csv"):
    for d in ['/kaggle/input/datasets/siddhanttimalsina/nepali-text-summarizer/',
              '/kaggle/input/datasets/sidsiddhant/nepali-text-summary-dataset/']:
        if os.path.exists(d + 'small.csv'):
            DATA_DIR = d; break
print(f"DATA_DIR: {DATA_DIR}")

all_data = pd.concat([
    pd.read_csv(DATA_DIR + "small.csv"),
    pd.read_csv(DATA_DIR + "medium.csv"),
    pd.read_csv(DATA_DIR + "large.csv"),
], ignore_index=True)
print(f"Total rows: {len(all_data)}")

train_df, temp_df = train_test_split(all_data, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

test_df[['content','summary']].to_csv('/kaggle/working/indicbart_test_raw.csv',
                                       index=False, encoding='utf-8')

# ---------- Tokenize with <2hi> label prefix ----------
MAX_INPUT_LEN, MAX_TARGET_LEN = 512, 128

def tokenize_fn(examples):
    inputs = tokenizer(examples['content'], max_length=MAX_INPUT_LEN,
                       truncation=True, padding=False)
    labels = []
    for s in examples['summary']:
        s_ids = tokenizer(s, max_length=MAX_TARGET_LEN - 1,
                          truncation=True, padding=False)['input_ids']
        labels.append([HI_ID] + s_ids)
    inputs['labels'] = labels
    return inputs

print("\nTokenizing...")
tok_train = Dataset.from_pandas(train_df[['content','summary']].reset_index(drop=True)) \
                .map(tokenize_fn, batched=True, remove_columns=['content','summary'])
tok_val   = Dataset.from_pandas(val_df[['content','summary']].reset_index(drop=True)) \
                .map(tokenize_fn, batched=True, remove_columns=['content','summary'])

assert tok_train[0]['labels'][0] == HI_ID, "Label prefix not applied"
print(f"First label[0]: {tok_train[0]['labels'][0]}  ✓")

# ---------- Training args ----------
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./indicbart-v3",
    eval_strategy="epoch", save_strategy="epoch",
    logging_strategy="steps", logging_steps=200,
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-4,
    warmup_steps=500,
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    fp16=True,
    predict_with_generate=True, generation_max_length=128,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    save_total_limit=2, report_to="none",
)

# ---------- Trainer (HF-version-proof init) ----------
try:
    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=tok_train, eval_dataset=tok_val,
        data_collator=data_collator,
        processing_class=tokenizer,
    )
    print("Trainer init: processing_class= (HF ≥ 4.46)")
except TypeError:
    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=tok_train, eval_dataset=tok_val,
        data_collator=data_collator,
    )
    trainer.tokenizer = tokenizer
    print("Trainer init: trainer.tokenizer= (HF < 4.46)")

# ---------- Train ----------
print("\n" + "="*60)
print("  TRAINING — 3 epochs, ~1.5 hours")
print("="*60)
t0 = time.time()
trainer.train()
print(f"\nTraining done in {(time.time()-t0)/3600:.2f}h")

# ---------- Sanity ----------
print("\n" + "="*60)
print("  SANITY GENERATION")
print("="*60)
sample = test_df.iloc[0]
inp = tokenizer(sample['content'], return_tensors='pt',
                max_length=512, truncation=True).to(model.device)
with torch.no_grad():
    out = model.generate(
        **inp, forced_bos_token_id=HI_ID,
        max_length=128, num_beams=5,
        no_repeat_ngram_size=4, repetition_penalty=2.0,
        length_penalty=2.0, early_stopping=True, min_length=20,
    )
pred = tokenizer.decode(out[0], skip_special_tokens=True)
print(f"REF  ({len(sample['summary'])} chars): {sample['summary'][:300]}")
print(f"PRED ({len(pred)} chars): {pred[:300]}")

# ---------- Save ----------
print("\nSaving model...")
model.save_pretrained("./indicbart-v3-final")
tokenizer.save_pretrained("./indicbart-v3-final")
shutil.make_archive("/kaggle/working/indicbart-v3", 'zip', "./indicbart-v3-final")
shutil.rmtree("./indicbart-v3", ignore_errors=True)
print("Saved indicbart-v3-final/ + indicbart-v3.zip")
print("\n✅ CELL 1 DONE. Run CELL 2.")

transformers 5.20.2
Device: cuda

Loading base model: ai4bharat/IndicBART


config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 1.90MB            

spiece.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

... (17 outputs left out)

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



Training done in 3.97h

  SANITY GENERATION
REF  (349 chars): स्वास्थ्य बीमा बोर्डले आगामी पाँच वर्षका लागि रणनीतिक योजना सार्वजनिक गरेको छ, जसमा संस्थागत सुदृढीकरण, ऐन नियमको सुधार, र नागरिकको सहभागिता बढाउने कुरालाई प्राथमिकता दिइएको छ। हाल २८ प्रतिशत नागरिक बीमामा आवद्ध भए पनि बीमितले गुणस्तरीय सेवा नपाएको गुनासो बढेको छ। सेवा प्रवाहमा सुधार र नवीकरण दर बढा
PRED (534 chars): <2hi> स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ, जसमा संरचनागत सुदृढीकरण र सुशासनलाई प्राथमिकता दिइएको छ। यसैगरी, स्वास्थ्य बीमा सन्चालनमा दिगो वित्तीय व्यवस्थापन, सूचीकृत स्वास्थ्य संस्थाबाट प्रवाह गरिने स्वास्थ्य सेवामा सुधार, बीमा व्यवस्थापन सूचना प्रणालीको सुदृढीकरण,

Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved indicbart-v3-final/ + indicbart-v3.zip

✅ CELL 1 DONE. Run CELL 2.


In [3]:
# ============================================================
# CELL 2: Batch inference on 5,053 test rows (with <2hi> cleanup)
# ============================================================
import os, time
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_DIR = "/kaggle/working/indicbart-v3-final"
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
HI_ID = tokenizer.convert_tokens_to_ids('<2hi>')
print(f"Loaded IndicBART-v3 | <2hi> id = {HI_ID}")

test_df = pd.read_csv('/kaggle/working/indicbart_test_raw.csv')
print(f"Test rows: {len(test_df)}")

class TestSet(Dataset):
    def __init__(self, frame): self.texts = frame['content'].astype(str).tolist()
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i]

def collate(batch):
    return tokenizer(batch, max_length=512, truncation=True,
                     padding=True, return_tensors='pt')

loader = DataLoader(TestSet(test_df), batch_size=8, shuffle=False,
                    collate_fn=collate, num_workers=2, pin_memory=True)

GEN_KWARGS = dict(
    forced_bos_token_id=HI_ID,
    max_length=128, num_beams=5,
    no_repeat_ngram_size=4, repetition_penalty=2.0,
    length_penalty=2.0, early_stopping=True, min_length=20,
)

def clean_pred(text):
    """Strip <2hi> / <pad> / <unk> artifacts from generated text"""
    for tok in ['<2hi>', '<pad>', '<unk>', '<s>', '</s>', '<2ne>']:
        text = text.replace(tok, '')
    return ' '.join(text.split()).strip()   # collapse whitespace

# Smoke test
print("Smoke test...")
first = next(iter(loader))
with torch.no_grad():
    out = model.generate(first['input_ids'].to(device),
                         attention_mask=first['attention_mask'].to(device),
                         **GEN_KWARGS)
prev = tokenizer.batch_decode(out, skip_special_tokens=True)
prev_clean = [clean_pred(p) for p in prev]
print(f"  RAW   len[0]={len(prev[0])} chars")
print(f"  CLEAN len[0]={len(prev_clean[0])} chars: {prev_clean[0][:150]}")
print("Smoke passed.\n")

# Full generation
all_preds = []
t0 = time.time()
for batch in tqdm(loader, desc="Generating IndicBART-v3"):
    with torch.no_grad():
        gen = model.generate(
            batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            **GEN_KWARGS,
        )
    decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
    all_preds.extend([clean_pred(d) for d in decoded])
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

pred_df = pd.DataFrame({
    'source':    test_df['content'].astype(str).tolist(),
    'reference': test_df['summary'].astype(str).tolist(),
    'predicted': all_preds,
})
pred_df.to_csv('/kaggle/working/test_predictions_indicbart.csv',
               index=False, encoding='utf-8')

empty = (pred_df['predicted'].str.strip() == '').sum()
avg = pred_df['predicted'].str.len().mean()
ref = pred_df['reference'].str.len().mean()
print(f"\nSaved test_predictions_indicbart.csv ({len(pred_df)} rows)")
print(f"  Empty: {empty} | Avg PRED: {avg:.0f} | Avg REF: {ref:.0f} | ratio: {avg/ref:.2f}")

print("\nFirst 2 samples:")
for i in range(2):
    print(f"\n--- sample {i+1} ---")
    print(f"REF : {pred_df['reference'].iloc[i][:200]}")
    print(f"PRED: {pred_df['predicted'].iloc[i][:200]}")
print("\n✅ CELL 2 DONE. Run CELL 3.")

Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Loaded IndicBART-v3 | <2hi> id = 64006
Test rows: 5053
Smoke test...
  RAW   len[0]=534 chars
  CLEAN len[0]=528 chars: स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ, जसमा संरचनागत सुदृढीकरण र सुशासनलाई प्राथमिकता दिइएको छ। यसैगरी, स्वास्थ्य बीम
Smoke passed.



Generating IndicBART-v3:   0%|          | 0/632 [00:00<?, ?it/s]


Done in 50.4 min

Saved test_predictions_indicbart.csv (5053 rows)
  Empty: 0 | Avg PRED: 322 | Avg REF: 302 | ratio: 1.06

First 2 samples:

--- sample 1 ---
REF : स्वास्थ्य बीमा बोर्डले आगामी पाँच वर्षका लागि रणनीतिक योजना सार्वजनिक गरेको छ, जसमा संस्थागत सुदृढीकरण, ऐन नियमको सुधार, र नागरिकको सहभागिता बढाउने कुरालाई प्राथमिकता दिइएको छ। हाल २८ प्रतिशत नागरिक ब
PRED: स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ, जसमा संरचनागत सुदृढीकरण र सुशासनलाई प्राथमिकता दिइएको छ। यसैगरी, स्वास्थ्य बीमा सन्चालनमा दिगो वित्तीय व्यवस्थापन, सूचीकृत स्वास

--- sample 2 ---
REF : गृहमन्त्री बालकृष्ण खाणले एमसीसी प्रस्ताव संसदमा पेश गर्न साइत नहेर्ने र सत्ता गठबन्धनमा साझा धारणा बनाउन पहल भइरहेको बताए। उनले यो अनुदान गरिबी निवारण, सडक, विद्युत, र रोजगारीका लागि भएको र यसको सही 
PRED: गृहमन्त्री बालकृष्ण खाणले एमसीसी प्रस्ताव संसदमा पेश गर्न साइत हेरिरहनु नपर्ने बताएका छन्। गोरखा उद्योग वाणिज्य संघको साधारणसभा उद्घाटन गर्न उनले सत्तागठवन्धनका दलहरूविच साझा धारणा बनाउन पहल भइरह

In [4]:
# ============================================================
# CELL 3: Final 5-metric scoring
# ============================================================
!pip install -q sacrebleu bert-score

import pandas as pd, numpy as np
from collections import Counter
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pred_df = pd.read_csv('/kaggle/working/test_predictions_indicbart.csv')

n_empty = int((pred_df['predicted'].astype(str).str.strip() == '').sum())
if n_empty > 0:
    pred_df.loc[pred_df['predicted'].astype(str).str.strip() == '', 'predicted'] = '।'

refs = pred_df['reference'].astype(str).tolist()
hyps = pred_df['predicted'].astype(str).tolist()
print(f"Evaluating {len(refs)} predictions (empty: {n_empty})\n")

bleu_flores = BLEU(tokenize='flores200', max_ngram_order=4).corpus_score(hyps, [refs]).score
bleu_13a    = BLEU(tokenize='13a',      max_ngram_order=4).corpus_score(hyps, [refs]).score

def ng(t, n): return [tuple(t[i:i+n]) for i in range(len(t)-n+1)]
def rn(pt, rt, n):
    p, r = ng(pt, n), ng(rt, n)
    if not p or not r: return 0.0
    ov = sum((Counter(p) & Counter(r)).values())
    pr, rc = ov/len(p), ov/len(r)
    return 0.0 if pr+rc == 0 else 2*pr*rc/(pr+rc)
def rl(pt, rt):
    m, n = len(pt), len(rt)
    if m == 0 or n == 0: return 0.0
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1]+1 if pt[i-1]==rt[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]; pr, rc = lcs/m, lcs/n
    return 0.0 if pr+rc == 0 else 2*pr*rc/(pr+rc)

r1 = float(np.mean([rn(h.split(), r.split(), 1) for h, r in zip(hyps, refs)]))
r2 = float(np.mean([rn(h.split(), r.split(), 2) for h, r in zip(hyps, refs)]))
rL = float(np.mean([rl(h.split(), r.split())       for h, r in zip(hyps, refs)]))

P, R, F1 = bert_score_fn(hyps, refs, lang='ne', verbose=False, device=device)
bs_p, bs_r, bs_f1 = P.mean().item(), R.mean().item(), F1.mean().item()

def set_jac(A, B):
    if not A and not B: return 1.0
    if not A or not B:  return 0.0
    return len(A & B) / len(A | B)
def jac_word(a, b): return set_jac(set(str(a).split()), set(str(b).split()))
def jac_char(a, b):
    a, b = str(a), str(b)
    return set_jac(set(a[i:i+2] for i in range(max(len(a)-1, 0))),
                   set(b[i:i+2] for i in range(max(len(b)-1, 0))))
jac_w = float(np.mean([jac_word(h, r) for h, r in zip(hyps, refs)]))
jac_c = float(np.mean([jac_char(h, r) for h, r in zip(hyps, refs)]))

print("=" * 60)
print("  IndicBART (3 epochs, all fixes) — FINAL SCORES")
print("=" * 60)
print(f"  SacreBLEU (BLEU-4, flores200)   : {bleu_flores:.4f}")
print(f"  SacreBLEU (BLEU-4, 13a)         : {bleu_13a:.4f}")
print(f"  ROUGE-1                         : {r1:.4f}")
print(f"  ROUGE-2                         : {r2:.4f}")
print(f"  ROUGE-L                         : {rL:.4f}")
print(f"  BERTScore Precision             : {bs_p:.4f}")
print(f"  BERTScore Recall                : {bs_r:.4f}")
print(f"  BERTScore F1                    : {bs_f1:.4f}")
print(f"  Jaccard (word-unigram)          : {jac_w:.4f}")
print(f"  Jaccard (char-bigram)           : {jac_c:.4f}")
print("=" * 60)
print(f"  Test samples                    : {len(refs)}")
print(f"  Empty predictions               : {n_empty}")
print(f"  Avg PRED length                 : {np.mean([len(h) for h in hyps]):.0f} chars")
print(f"  Avg REF  length                 : {np.mean([len(r) for r in refs]):.0f} chars")
print("=" * 60)

pd.Series({
    'model': 'IndicBART-3ep-fixed',
    'sacrebleu_flores200': round(bleu_flores, 4),
    'sacrebleu_13a':       round(bleu_13a, 4),
    'rouge1':              round(r1, 4),
    'rouge2':              round(r2, 4),
    'rougeL':              round(rL, 4),
    'bertscore_p':         round(bs_p, 4),
    'bertscore_r':         round(bs_r, 4),
    'bertscore_f1':        round(bs_f1, 4),
    'jaccard_word':        round(jac_w, 4),
    'jaccard_char':        round(jac_c, 4),
    'n_samples':           len(refs),
    'n_empty':             n_empty,
}).to_csv('/kaggle/working/final_scores_indicbart.csv')
print("\nSaved final_scores_indicbart.csv")
print("\n⚠️  Click 'Save Version' to persist!")

Evaluating 5053 predictions (empty: 0)



config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  IndicBART (3 epochs, all fixes) — FINAL SCORES
  SacreBLEU (BLEU-4, flores200)   : 29.8325
  SacreBLEU (BLEU-4, 13a)         : 17.3407
  ROUGE-1                         : 0.3910
  ROUGE-2                         : 0.2167
  ROUGE-L                         : 0.3252
  BERTScore Precision             : 0.7751
  BERTScore Recall                : 0.7942
  BERTScore F1                    : 0.7839
  Jaccard (word-unigram)          : 0.2630
  Jaccard (char-bigram)           : 0.5168
  Test samples                    : 5053
  Empty predictions               : 0
  Avg PRED length                 : 322 chars
  Avg REF  length                 : 302 chars

Saved final_scores_indicbart.csv

⚠️  Click 'Save Version' to persist!
